# Four-model workplace evaluation

This notebook runs the 35 workplace inputs through the two fine-tuned T5 models and the two activation-steering configurations used by the Gradio demo. It saves one long-form prediction CSV, evaluates every response with the existing SecondThought metrics, and produces a model-level summary.

The steering models use a fixed alpha so their results are comparable. Formality evaluation is intentionally marked **Pending** until an independent formality evaluator is developed.

In [1]:
from pathlib import Path
from time import perf_counter
from datetime import UTC, datetime
import json
import sys

import pandas as pd

PROJECT_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "pyproject.toml").is_file()),
    Path.cwd(),
)
SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from secondthought.evaluation import (
    DetoxifyToxicityScorer,
    SentenceTransformerSimilarityScorer,
    evaluate_records,
    format_evaluated_row,
)
from secondthought.fine_tuned_rewriter import (
    FINE_TUNED_VARIANTS,
    FineTunedT5Rewriter,
)
from secondthought.inference_from_jsonl import read_inputs
from secondthought.steering_rewriter import STEERING_VARIANTS, SteeredQwenRewriter

INPUT_PATH = PROJECT_ROOT / "data/workplace_eval/input.jsonl"
RESULTS_DIRECTORY = PROJECT_ROOT / "testing/results"
PREDICTIONS_PATH = RESULTS_DIRECTORY / "four_model_predictions.csv"
EVALUATED_PATH = RESULTS_DIRECTORY / "four_model_evaluated.csv"
SUMMARY_PATH = RESULTS_DIRECTORY / "four_model_summary.csv"

STEERING_ALPHA = 0.5
RESUME_GENERATION = True
RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Steering alpha:", STEERING_ALPHA)

/Users/nandanprince/Desktop/NLP_project/SecondThought/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /Users/nandanprince/Desktop/NLP_project/SecondThought
Steering alpha: 0.5


## 1. Load and validate the evaluation inputs

The source file must contain unique, non-empty `id` and `original_text` fields.

In [2]:
input_records = read_inputs(INPUT_PATH)
inputs = pd.DataFrame(input_records)
assert len(inputs) == 35, f"Expected 35 inputs, found {len(inputs)}"
assert inputs["id"].is_unique
print(f"Loaded {len(inputs)} inputs from {INPUT_PATH.relative_to(PROJECT_ROOT)}")
display(inputs.head())

Loaded 35 inputs from data/workplace_eval/input.jsonl


,id,original_text
0,input_001,"Seriously, Maya, this invoice is a damn mess. ..."
1,input_002,You dumped 18 broken laptops on my desk like a...
2,input_003,"I'm sick of your bullshit excuses, Daniel. The..."
3,input_004,Who the hell sent the boxes to 47 Willow Lane?...
4,input_005,This lazy crap has cost us 2 hours already. Re...


## 2. Generate all four responses

Generation is deterministic for the current implementations: Qwen uses greedy decoding and both T5 models use four-beam decoding without sampling. Alpha is passed only to the two activation-steered responses. The CSV is checkpointed after every input so an interrupted run can resume.

In [3]:
MODEL_LABELS = {
    "detox_t5": "Detox Fine Tuned",
    "formalize_t5": "Formality Fine Tuned",
    "alternative_layer4_final": "Formality Steering - Layer 4",
    "negative_layer11_mean": "Formality Steering - Layer 11",
}
EXPECTED_MODEL_NAMES = list(MODEL_LABELS.values())
EXPECTED_ROWS = len(input_records) * len(EXPECTED_MODEL_NAMES)

if RESUME_GENERATION and PREDICTIONS_PATH.is_file():
    predictions = pd.read_csv(PREDICTIONS_PATH).to_dict(orient="records")
    print(f"Resuming from {len(predictions)} saved rows")
else:
    predictions = []

completed = {(row["id"], row["model_name"]) for row in predictions}
needs_generation = any(
    (record["id"], model_name) not in completed
    for record in input_records
    for model_name in EXPECTED_MODEL_NAMES
)

if needs_generation:
    print("Loading the four local generation backends...")
    steering_rewriter = SteeredQwenRewriter()
    fine_tuned_rewriter = FineTunedT5Rewriter()

    for input_number, record in enumerate(input_records, start=1):
        record_id = record["id"]
        text = record["original_text"]

        missing_fine_tuned = [
            variant for variant in FINE_TUNED_VARIANTS
            if (record_id, MODEL_LABELS[variant.key]) not in completed
        ]
        if missing_fine_tuned:
            started = perf_counter()
            generated = fine_tuned_rewriter.rewrite_all(text)
            elapsed_ms = (perf_counter() - started) * 1000
            per_response_ms = elapsed_ms / len(FINE_TUNED_VARIANTS)
            for variant in missing_fine_tuned:
                predictions.append({
                    **record,
                    "model_name": MODEL_LABELS[variant.key],
                    "model_key": variant.key,
                    "alpha": pd.NA,
                    "rewritten_text": generated[variant.key],
                    "latency_ms": round(per_response_ms, 2),
                    "timestamp_utc": datetime.now(UTC).isoformat(),
                })
                completed.add((record_id, MODEL_LABELS[variant.key]))

        missing_steering = [
            variant for variant in STEERING_VARIANTS
            if (record_id, MODEL_LABELS[variant.key]) not in completed
        ]
        if missing_steering:
            started = perf_counter()
            generated = steering_rewriter.rewrite_all(text, STEERING_ALPHA)
            elapsed_ms = (perf_counter() - started) * 1000
            per_response_ms = elapsed_ms / len(STEERING_VARIANTS)
            for variant in missing_steering:
                predictions.append({
                    **record,
                    "model_name": MODEL_LABELS[variant.key],
                    "model_key": variant.key,
                    "alpha": STEERING_ALPHA,
                    "rewritten_text": generated[variant.key],
                    "latency_ms": round(per_response_ms, 2),
                    "timestamp_utc": datetime.now(UTC).isoformat(),
                })
                completed.add((record_id, MODEL_LABELS[variant.key]))

        prediction_frame = pd.DataFrame(predictions).sort_values(["id", "model_name"])
        prediction_frame.to_csv(PREDICTIONS_PATH, index=False)
        print(f"[{input_number:02d}/{len(input_records)}] {record_id} — {len(prediction_frame)}/{EXPECTED_ROWS} rows")
else:
    print("Prediction CSV is already complete; generation skipped.")

prediction_frame = pd.read_csv(PREDICTIONS_PATH)
assert len(prediction_frame) == EXPECTED_ROWS
assert not prediction_frame.duplicated(["id", "model_name"]).any()
assert set(prediction_frame["model_name"]) == set(EXPECTED_MODEL_NAMES)
print("Saved predictions:", PREDICTIONS_PATH)
display(prediction_frame[["id", "model_name", "alpha", "original_text", "rewritten_text"]].head(8))

Resuming from 140 saved rows
Prediction CSV is already complete; generation skipped.
Saved predictions: /Users/nandanprince/Desktop/NLP_project/SecondThought/testing/results/four_model_predictions.csv


,id,model_name,alpha,original_text,rewritten_text
0,input_001,Detox Fine Tuned,NaN,"Seriously, Maya, this invoice is a damn mess. ...","Seriously, Maya, this invoice is a mess. Chang..."
1,input_001,Formality Fine Tuned,NaN,"Seriously, Maya, this invoice is a damn mess. ...","Honestly, Maya, this invoice is a mess. Change..."
2,input_001,Formality Steering - Layer 11,0.5,"Seriously, Maya, this invoice is a damn mess. ...","Seriously, Maya, change the total from $480 to..."
3,input_001,Formality Steering - Layer 4,0.5,"Seriously, Maya, this invoice is a damn mess. ...","Seriously, Maya, change the total from $480 to..."
4,input_002,Detox Fine Tuned,NaN,You dumped 18 broken laptops on my desk like a...,You dumped 18 broken laptops on my desk. Get t...
5,input_002,Formality Fine Tuned,NaN,You dumped 18 broken laptops on my desk like a...,You dumped 18 broken laptops on my desk like a...
6,input_002,Formality Steering - Layer 11,0.5,You dumped 18 broken laptops on my desk like a...,"Dumped 18 broken laptops on your desk, like an..."
7,input_002,Formality Steering - Layer 4,0.5,You dumped 18 broken laptops on my desk like a...,"You dropped 18 broken laptops on my desk, like..."


## 3. Evaluate every generated response

The existing evaluation pipeline computes:

- **Toxicity:** Detoxify scores the input and output. Lower output toxicity and larger reduction are better.
- **Semantic similarity:** cosine similarity from `all-mpnet-base-v2`. Higher values indicate closer overall meaning, but do not guarantee that every fact or requested action is preserved.
- **Critical-value preservation:** deterministic recall for numbers, money, dates, times, percentages, measurements, URLs, emails, ticket IDs, and technical identifiers. Unsupported values identify newly introduced structured facts.
- **Formality:** pending; no formality metric is calculated in this notebook.

In [4]:
print("Loading Detoxify and sentence-transformer evaluators...")
toxicity_scorer = DetoxifyToxicityScorer(model_type="original", device="cpu")
similarity_scorer = SentenceTransformerSimilarityScorer(
    model_name="sentence-transformers/all-mpnet-base-v2",
    device="cpu",
)

prediction_records = prediction_frame.to_dict(orient="records")
metric_rows = evaluate_records(
    prediction_records,
    toxicity_scorer=toxicity_scorer,
    similarity_scorer=similarity_scorer,
)
evaluated_records = [
    format_evaluated_row(prediction, metrics)
    for prediction, metrics in zip(prediction_records, metric_rows, strict=True)
]
evaluated = pd.DataFrame(evaluated_records)
evaluated.to_csv(EVALUATED_PATH, index=False)
assert len(evaluated) == EXPECTED_ROWS
print(f"Saved {len(evaluated)} evaluated responses to {EVALUATED_PATH}")
display(evaluated.head())

Loading Detoxify and sentence-transformer evaluators...


Saved 140 evaluated responses to /Users/nandanprince/Desktop/NLP_project/SecondThought/testing/results/four_model_evaluated.csv


,id,original_text,rewritten_text,semantic_similarity,toxicity_original,toxicity_rewritten,toxicity_reduction,toxicity_relative_reduction,critical_value_count,critical_value_preserved_count,...,word_overlap,edit_ratio,length_ratio,exact_match,number_recall,model_name,model_key,alpha,latency_ms,timestamp_utc
0,input_001,"Seriously, Maya, this invoice is a damn mess. ...","Seriously, Maya, this invoice is a mess. Chang...",0.997747,0.736666,0.041451,0.695214,0.943731,2,2,...,0.9474,0.0256,0.9500,False,1.0,Detox Fine Tuned,detox_t5,NaN,701.99,2026-10-06T19:15:28.045582+00:00
1,input_001,"Seriously, Maya, this invoice is a damn mess. ...","Honestly, Maya, this invoice is a mess. Change...",0.990220,0.736666,0.035064,0.701601,0.952401,2,2,...,0.8500,0.0722,0.9400,False,1.0,Formality Fine Tuned,formalize_t5,NaN,701.99,2026-10-06T19:15:28.045593+00:00
2,input_001,"Seriously, Maya, this invoice is a damn mess. ...","Seriously, Maya, change the total from $480 to...",0.922376,0.736666,0.001330,0.735336,0.998195,2,2,...,0.5500,0.2626,0.7900,False,1.0,Formality Steering - Layer 11,negative_layer11_mean,0.5,1711.88,2026-10-06T19:15:31.469385+00:00
3,input_001,"Seriously, Maya, this invoice is a damn mess. ...","Seriously, Maya, change the total from $480 to...",0.922376,0.736666,0.001330,0.735336,0.998195,2,2,...,0.5500,0.2626,0.7900,False,1.0,Formality Steering - Layer 4,alternative_layer4_final,0.5,1711.88,2026-10-06T19:15:31.469376+00:00
4,input_002,You dumped 18 broken laptops on my desk like a...,You dumped 18 broken laptops on my desk. Get t...,0.983618,0.963497,0.020671,0.942827,0.978546,3,3,...,0.8500,0.0864,0.8409,False,1.0,Detox Fine Tuned,detox_t5,NaN,349.73,2026-10-06T19:15:32.171357+00:00


## 4. Model-level summary

Critical-value recall is micro-averaged: all preserved source values divided by all source values. The toxicity figures and semantic similarity are arithmetic means across the 35 responses from each model.

In [5]:
summary_rows = []
for model_name in EXPECTED_MODEL_NAMES:
    group = evaluated.loc[evaluated["model_name"] == model_name]
    critical_total = int(group["critical_value_count"].sum())
    critical_preserved = int(group["critical_value_preserved_count"].sum())
    summary_rows.append({
        "model_name": model_name,
        "examples": len(group),
        "mean_toxicity_original": group["toxicity_original"].mean(),
        "mean_toxicity_rewritten": group["toxicity_rewritten"].mean(),
        "mean_toxicity_reduction": group["toxicity_reduction"].mean(),
        "mean_relative_toxicity_reduction": group["toxicity_relative_reduction"].mean(),
        "mean_semantic_similarity": group["semantic_similarity"].mean(),
        "critical_values_preserved": critical_preserved,
        "critical_values_total": critical_total,
        "critical_value_micro_recall": critical_preserved / critical_total if critical_total else 1.0,
        "missing_critical_values": int(group["critical_value_missing_count"].sum()),
        "unsupported_critical_values": int(group["critical_value_unsupported_count"].sum()),
        "critical_value_exact_rate": group["critical_value_exact"].mean(),
        "mean_latency_ms": group["latency_ms"].mean(),
        "formality": "Pending",
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(SUMMARY_PATH, index=False)
slide_table = summary[[
    "model_name",
    "mean_toxicity_original",
    "mean_toxicity_rewritten",
    "mean_relative_toxicity_reduction",
    "mean_semantic_similarity",
    "critical_values_preserved",
    "critical_values_total",
    "critical_value_micro_recall",
    "unsupported_critical_values",
    "formality",
]].copy()
slide_table["toxicity"] = slide_table.apply(
    lambda row: f'{row["mean_toxicity_original"]:.3f} → {row["mean_toxicity_rewritten"]:.3f}',
    axis=1,
)
slide_table["relative_toxicity_reduction"] = slide_table["mean_relative_toxicity_reduction"].map(lambda value: f"{value:.1%}")
slide_table["semantic_similarity"] = slide_table["mean_semantic_similarity"].map(lambda value: f"{value:.3f}")
slide_table["critical_values"] = slide_table.apply(
    lambda row: f'{int(row["critical_values_preserved"])}/{int(row["critical_values_total"])} ({row["critical_value_micro_recall"]:.1%})',
    axis=1,
)
slide_table = slide_table[[
    "model_name",
    "toxicity",
    "relative_toxicity_reduction",
    "semantic_similarity",
    "critical_values",
    "unsupported_critical_values",
    "formality",
]]
print("Saved summary:", SUMMARY_PATH)
display(slide_table)

Saved summary: /Users/nandanprince/Desktop/NLP_project/SecondThought/testing/results/four_model_summary.csv


,model_name,toxicity,relative_toxicity_reduction,semantic_similarity,critical_values,unsupported_critical_values,formality
0,Detox Fine Tuned,0.604 → 0.056,83.9%,0.967,45/45 (100.0%),0,Pending
1,Formality Fine Tuned,0.604 → 0.366,39.4%,0.977,43/45 (95.6%),3,Pending
2,Formality Steering - Layer 4,0.604 → 0.304,53.7%,0.880,42/45 (93.3%),1,Pending
3,Formality Steering - Layer 11,0.604 → 0.164,74.5%,0.863,40/45 (88.9%),2,Pending


## 5. Inspect the most important failures

Aggregate means can conceal deletions, substitutions, and invented values. The following table prioritizes low-similarity responses and any critical-value failures for manual review.

In [6]:
review = evaluated.assign(
    has_critical_value_issue=(
        (evaluated["critical_value_missing_count"] > 0)
        | (evaluated["critical_value_unsupported_count"] > 0)
    )
).sort_values(
    ["has_critical_value_issue", "semantic_similarity"],
    ascending=[False, True],
)
display(review[[
    "id",
    "model_name",
    "original_text",
    "rewritten_text",
    "semantic_similarity",
    "toxicity_rewritten",
    "critical_value_missing",
    "critical_value_unsupported",
]].head(20))

,id,model_name,original_text,rewritten_text,semantic_similarity,toxicity_rewritten,critical_value_missing,critical_value_unsupported
59,input_015,Formality Steering - Layer 4,"Leah, stop talking out of your ass. We have 4 ...","Leah, please stop talking and let us know if t...",0.537518,0.000574,"[{""kind"":""number"",""raw"":""4"",""normalized"":""4"",""...",[]
58,input_015,Formality Steering - Layer 11,"Leah, stop talking out of your ass. We have 4 ...","Leah, please stop talking and let us know if t...",0.673288,0.000561,"[{""kind"":""number"",""raw"":""4"",""normalized"":""4"",""...",[]
34,input_009,Formality Steering - Layer 11,I don't care about your pathetic excuses. Refu...,Refund the $79. 50 charge to Jordan today.,0.797139,0.000811,"[{""kind"":""money"",""raw"":""$79.50"",""normalized"":""...","[{""kind"":""money"",""raw"":""$79"",""normalized"":""USD..."
57,input_015,Formality Fine Tuned,"Leah, stop talking out of your ass. We have 4 ...",Please stop talking out of your mouth. We have...,0.838823,0.306855,[],"[{""kind"":""number_word"",""raw"":""nine"",""normalize..."
18,input_005,Formality Steering - Layer 11,This lazy crap has cost us 2 hours already. Re...,Restart the scanner and call Priya if it fails...,0.859237,0.002136,"[{""kind"":""measurement"",""raw"":""2 hours"",""normal...",[]
54,input_014,Formality Steering - Layer 11,This useless update made the page take 12 seco...,Roll this useless update before the demo and s...,0.860095,0.027196,"[{""kind"":""measurement"",""raw"":""12 seconds"",""nor...",[]
10,input_003,Formality Steering - Layer 11,"I'm sick of your bullshit excuses, Daniel. The...","I'm tired of your empty promises, Daniel. Your...",0.866287,0.001583,"[{""kind"":""relative_date"",""raw"":""Friday"",""norma...",[]
81,input_021,Formality Fine Tuned,Quit screwing around with the schedule. Luis s...,Luis starts at 8:30 p.m. and Erin takes over a...,0.908818,0.001188,"[{""kind"":""time"",""raw"":""8:30"",""normalized"":""08:...","[{""kind"":""time"",""raw"":""8:30 p.m"",""normalized"":..."
67,input_017,Formality Steering - Layer 4,Your careless packing is ridiculous. Put 2 kg ...,Your packing is careless and unreasonable. Pac...,0.931854,0.013865,"[{""kind"":""number"",""raw"":""200"",""normalized"":""20...","[{""kind"":""measurement"",""raw"":""200 grams"",""norm..."
51,input_013,Formality Steering - Layer 4,I'm not cleaning up your damn mess for the fou...,I'm not cleaning up your damn mess again. Send...,0.974149,0.652893,"[{""kind"":""number_word"",""raw"":""fourth"",""normali...",[]


## Results report

All four models completed the same 35-message workplace set. The steering results use **alpha = +0.5**. Mean input toxicity is 0.604 for every model because the source dataset is shared.

| Model | Mean toxicity (input → output) | Mean relative reduction | Mean semantic similarity | Critical values preserved | Unsupported values | Formality |
|---|---:|---:|---:|---:|---:|---|
| Detox Fine Tuned | 0.604 → **0.056** | **83.9%** | **0.967** | **45/45 (100%)** | **0** | Pending |
| Formality Fine Tuned | 0.604 → 0.366 | 39.4% | **0.977** | 43/45 (95.6%) | 3 | Pending |
| Formality Steering — Layer 4 | 0.604 → 0.304 | 53.7% | 0.880 | 42/45 (93.3%) | 1 | Pending |
| Formality Steering — Layer 11 | 0.604 → **0.164** | **74.5%** | 0.863 | 40/45 (88.9%) | 2 | Pending |

### Interpretation

- **Detox Fine Tuned performs best on the metrics currently available.** It has the lowest mean output toxicity, retains all 45 extracted critical values, introduces no unsupported structured values, and maintains high semantic similarity. This is expected because both the model and the evaluation set emphasize detoxification.
- **Formality Fine Tuned has the highest mean semantic similarity**, but it removes less toxicity and loses two critical values. Its high similarity score demonstrates why embedding similarity must be read together with exact preservation checks.
- **Layer 11 has the stronger toxicity effect of the two steering interventions** at alpha +0.5: mean output toxicity falls to 0.164 compared with 0.304 for Layer 4. The cost is lower semantic similarity and five missing critical values.
- **Layer 4 is the more conservative steering intervention** on this set. It changes toxicity less aggressively than Layer 11 but preserves more critical values and has slightly higher semantic similarity.
- These results do **not** establish which model formalizes best. A model can preserve meaning by barely rewriting, and Detoxify does not measure formality. Overall model selection should remain pending until an independent formality measure and human review are added.

## Interpretation guardrails

- Detoxify was trained for toxicity detection, not workplace professionalism or formality.
- Embedding similarity can miss changed negation, responsibility, urgency, or requested actions.
- Critical-value preservation checks structured literals, not every factual relationship.
- The steering results on this page apply only to the configured alpha.
- Formality remains pending and must be evaluated independently rather than inferred from toxicity reduction.